**Cell 1 — Read the source CSV:** loads the raw CSV from the volume with header + inferred schema into a lazy DataFrame.

In [0]:
electronics_df = spark.read.csv(
    "/Volumes/db_final/bronze/raw_data/electronics_sales_initial.csv",
    header=True,
    inferSchema=True
)

**Cell 2 — Preview raw data:** first look at the 100 raw records via `display()`.

In [0]:
display(electronics_df)

**Cell 3 — Inspect schema:** confirms the inferred types — date, int, double, strings.

In [0]:
electronics_df.printSchema()

**Cell 4 — Source count:** 100 rows — the reconciliation baseline.

In [0]:
print("Source Count =", electronics_df.count())

**Cell 5 — Import Spark functions:** makes `col`, `lit`, `current_timestamp` available.

In [0]:
from pyspark.sql.functions import *

**Cell 6 — Build bronze_df:** adds `source_file`, `batch_id`, `ingestion_timestamp` audit columns.

In [0]:
bronze_df = electronics_df \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("batch_id", lit("BATCH_001")) \
    .withColumn("ingestion_timestamp", current_timestamp())

**Cell 7 — Preview bronze_df:** the enriched data exactly as it will be written.

In [0]:
display(bronze_df)

**Cell 8 — Write raw bronze table:** all 100 rows → `db_final.bronze.electronics_sales_bronze` (Delta, overwrite).

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_final.bronze.electronics_sales_bronze")

**Cell 9 — Validate records:** tags `rejection_reason` (missing IDs, bad quantity/price/discount/warranty, invalid status/date) and rejects duplicate `sale_id`s.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

validated_df = bronze_df \
    .withColumn(
        "rejection_reason",
        when(
            col("sale_id").isNull() | (trim(col("sale_id")) == ""),
            "SALE_ID_MISSING"
        ).when(
            col("customer_id").isNull() | (trim(col("customer_id")) == ""),
            "CUSTOMER_ID_MISSING"
        ).when(
            col("product_name").isNull() | (trim(col("product_name")) == ""),
            "PRODUCT_NAME_MISSING"
        ).when(
            col("brand").isNull() | (trim(col("brand")) == ""),
            "BRAND_MISSING"
        ).when(
            col("quantity") <= 0,
            "INVALID_QUANTITY"
        ).when(
            col("unit_price") <= 0,
            "INVALID_UNIT_PRICE"
        ).when(
            col("discount") < 0,
            "INVALID_DISCOUNT"
        ).when(
            col("warranty_years") < 0,
            "INVALID_WARRANTY"
        ).when(
            to_date(col("sale_date")).isNull(),
            "INVALID_SALE_DATE"
        ).when(
            ~col("sale_status").isin(
                "COMPLETED",
                "RETURNED",
                "CANCELLED"
            ),
            "INVALID_STATUS"
        )
    )

# Duplicate handling: keep ONE valid record per sale_id.
# Only rows that pass all quality checks compete for the sale_id;
# a valid occurrence after the first is rejected as DUPLICATE_SALE_ID.

dup_window = Window.partitionBy("sale_id").orderBy(
    col("rejection_reason").isNull().desc(),    
    col("sale_date").asc()
)

validated_df = validated_df \
    .withColumn(
        "duplicate_flag",
        when(
            col("rejection_reason").isNull()
            & (row_number().over(dup_window) > 1),
            lit("Y")
        )
    ) \
    .withColumn(
        "rejection_reason",
        when(
            col("duplicate_flag") == "Y",
            "DUPLICATE_SALE_ID"
        ).otherwise(col("rejection_reason"))
    )

**Cell 10 — Split valid/rejected:** NULL reason → `valid_df`, otherwise `rejected_df` (75 / 25).

In [0]:
valid_df = validated_df.filter(col("rejection_reason").isNull())

rejected_df = validated_df.filter(col("rejection_reason").isNotNull())

**Cell 11 — Write valid table:** 75 clean rows → `electronic_sales.bronze.electronics_sales_valid`.

In [0]:
valid_df = validated_df.filter(col("rejection_reason").isNull())

valid_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.bronze.electronics_sales_valid"
    )

**Cell 12 — Write rejected table:** 25 bad rows → `electronic_sales.audit.electronics_sales_rejected`.

In [0]:
rejected_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "electronic_sales.audit.electronics_sales_rejected"
    )

**Cell 13 — Preview tables:** reads both written tables back — round-trip check.

In [0]:
display(
    spark.table(
        "electronic_sales.bronze.electronics_sales_valid"
    )
)

display(
    spark.table(
        "electronic_sales.audit.electronics_sales_rejected"
    )
)

**Cell 14 — Reconciliation:** source = valid + rejected, and only valid rows may enter Silver.

In [0]:
source_count = bronze_df.count()
print("Source Count :", source_count)

In [0]:
valid_count = valid_df.count()
print("Valid Count :", valid_count)

In [0]:
rejected_count = rejected_df.count()
print("Rejected Count :", rejected_count)

In [0]:
# Source Count = Valid Count + Rejected Count.

print(
    "\nValidation Check :",
    f"{source_count} = {valid_count} + {rejected_count}"
)


In [0]:
# Status

print(
    "\nReconciliation Status :",
    reconciliation_status
)

In [0]:
# •	Confirm that rejected records do not enter Silver

silver_source_count = valid_df.count()

if silver_source_count == valid_count:
    print(
        "\nSilver Validation : PASS - Only valid records entered Silver"
    )
else:
    print(
        "\nSilver Validation : FAIL - Rejected records entered Silver"
    )

**Bronze layer complete:** 100 raw → 75 valid + 25 rejected, reconciliation PASS ✅